# Capstone Two Modeling

### Goal: 
The goal of this notebook is to build and evaluate multiple machine learning models to predict whether a shipment arrives on time using `Reached.on.Time_Y.N` as the target variable.

This notebook uses the preprocessed training and testing datasets created in the previous assignment. For each model, hyperparameter tuning is performed using `GridSearchCV` with cross validation. After tuning, model performance is compared using several evaluation metrics to identify the best performing model.

In [19]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay

import warnings
warnings.filterwarnings("ignore")

## Load Cleaned Dataset

The cleaned dataset generated during the data preprocessing stage is loaded below.

In [20]:
df = pd.read_csv("cleaned_shipping_data.csv")

df.head()

,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1
3,4,B,Flight,3,3,176,4,medium,M,10,1177,1
4,5,C,Flight,2,2,184,3,medium,F,46,2484,1


In [21]:
df.shape

(10999, 12)

## Define Features and Target Variable

In [22]:
X = df.drop('Reached.on.Time_Y.N', axis=1)
y = df['Reached.on.Time_Y.N']

## Convert Categorical Variables

In [23]:
X = pd.get_dummies(X, drop_first=True)

X.head()

,ID,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Discount_offered,Weight_in_gms,Warehouse_block_B,Warehouse_block_C,Warehouse_block_D,Warehouse_block_F,Mode_of_Shipment_Road,Mode_of_Shipment_Ship,Product_importance_low,Product_importance_medium,Gender_M
0,1,4,2,177,3,44,1233,False,False,True,False,False,False,True,False,False
1,2,4,5,216,2,59,3088,False,False,False,True,False,False,True,False,True
2,3,2,2,183,4,48,3374,False,False,False,False,False,False,True,False,True
3,4,3,3,176,4,10,1177,True,False,False,False,False,False,False,True,True
4,5,2,2,184,3,46,2484,False,True,False,False,False,False,False,True,False


## Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

## Model Evaluation Function

This helper function calculates performance metrics for each trained model.

In [ ]:
def evaluate_model(model, X_test, y_test, name):

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])

    return {
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC AUC": roc_auc
    }

## Model 1: Logistic Regression with Hyperparameter Tuning

In [ ]:
log_model = LogisticRegression(max_iter=10000)

log_param_grid = {
    "C": [0.01,0.1,1,10],
    "solver": ["liblinear","lbfgs"]
}

grid_log = GridSearchCV(
    log_model,
    log_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_log.fit(X_train,y_train)

best_log_model = grid_log.best_estimator_

print("Best Parameters:", grid_log.best_params_)

In [ ]:
log_results = evaluate_model(best_log_model,X_test,y_test,"Logistic Regression")

log_results

## Model 2: Decision Tree with Hyperparameter Tuning

In [ ]:
tree_model = DecisionTreeClassifier()

tree_param_grid = {
    "max_depth":[3,5,10,None],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4],
    "criterion":["gini","entropy"]
}

grid_tree = GridSearchCV(
    tree_model,
    tree_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_tree.fit(X_train,y_train)

best_tree_model = grid_tree.best_estimator_

print("Best Parameters:", grid_tree.best_params_)

In [ ]:
tree_results = evaluate_model(best_tree_model,X_test,y_test,"Decision Tree")

tree_results

# Model 3: Random Forest with Hyperparameter Tuning

In [ ]:
rf_model = RandomForestClassifier()

rf_param_grid = {
    "n_estimators":[100,200],
    "max_depth":[5,10,None],
    "min_samples_split":[2,5],
    "min_samples_leaf":[1,2],
    "max_features":["sqrt","log2"]
}

grid_rf = GridSearchCV(
    rf_model,
    rf_param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_rf.fit(X_train,y_train)

best_rf_model = grid_rf.best_estimator_

print("Best Parameters:", grid_rf.best_params_)

In [ ]:
rf_results = evaluate_model(best_rf_model,X_test,y_test,"Random Forest")

rf_results

## Model Performance Comparison

In [ ]:
comparison = pd.DataFrame([log_results,tree_results,rf_results])

comparison = comparison.sort_values("F1 Score",ascending=False)

comparison

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(data=comparison,x="Model",y="F1 Score")

plt.title("Model Comparison (F1 Score)")
plt.show()

In [ ]:
best_model_name = comparison.iloc[0]["Model"]

print("Best Model:", best_model_name)

## ROC Curve Comparison

In [ ]:
plt.figure(figsize=(8,6))

RocCurveDisplay.from_estimator(best_log_model,X_test,y_test)
RocCurveDisplay.from_estimator(best_tree_model,X_test,y_test)
RocCurveDisplay.from_estimator(best_rf_model,X_test,y_test)

plt.title("ROC Curve Comparison")
plt.show()

## Conclusion

# Conclusion

Three machine learning models were developed and optimized using GridSearchCV: Logistic Regression, Decision Tree, and Random Forest.

After tuning and evaluating each model using multiple metrics, the models were compared using F1 score and ROC AUC. The model with the highest F1 score was selected as the best performing model for predicting shipment delivery timeliness.

Hyperparameter tuning significantly improves model performance by identifying optimal parameter combinations, ensuring that the comparison between algorithms is fair and reliable.